# Как BERT и GPT работают с текстом

Можно ли угадать оценку по тексту отзыва? Будем предсказывать одну из трёх групп звёзд (1–2, 3 или 4–5). Сравним признаки из слов и из готового BERT.

Затем проверим, как эти способы выбирают окончание короткой истории. Здесь пригодится и GPT. Она сможет оценить готовые варианты и написать своё продолжение.

## 0. Подготовка

In [1]:
!pip install -q "transformers>=4.45,<5" "scikit-learn>=1.4,<2" pandas scipy

Библиотеки уже установлены в проверочном окружении


In [2]:
import logging
import random
import re
import warnings

# Приглушаем служебные предупреждения библиотек.
warnings.filterwarnings("ignore", message=r"IProgress not found|TypedStorage is deprecated")
logging.getLogger("transformers.modeling_utils").addFilter(
    lambda record: not record.getMessage().startswith("BertForMaskedLM has generative capabilities")
)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from IPython.display import display
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoModelForMaskedLM, AutoTokenizer, set_seed

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Устройство:", DEVICE)

Устройство: mps


In [4]:
WB_REVIEWS_URL = (
    "https://huggingface.co/datasets/Hplss/wb-review-dataset/"
    "resolve/main/filtered.csv"
)
LABEL_NAMES = np.array(["отрицательный", "нейтральный", "положительный"])

reviews = pd.read_csv(WB_REVIEWS_URL)
parts = reviews[["text", "pros", "cons"]].fillna("").astype(str)
reviews["input"] = parts.agg(" ".join, axis=1).str.strip()
reviews = reviews[reviews["input"].str.len() > 10].copy()
reviews["label"] = np.select(
    [reviews["rating"] <= 2, reviews["rating"] == 3], [0, 1], default=2
)
reviews = reviews.reset_index(drop=True)

display(reviews[["rating", "category_label", "input"]].head(3))
print("Всего отзывов:", len(reviews))

,rating,category_label,input
0,5,Электроника,"Звук не плохой, максимальная громкость высокая..."
1,5,Электроника,"Ничего. Подключалась 3 дня еле подключилась, н..."
2,5,Электроника,Сразу после первого подключения не работал вто...


Всего отзывов: 27248


Отзывы об одном товаре часто похожи. Если часть таких отзывов попадёт в обучение, а часть в проверку, модели может хватить названия товара. Поэтому все отзывы с одним `product_id` оставляем в одной части данных. Сейчас учимся на train и сравниваем варианты на validation. Отдельный test пригодится на практике.

In [5]:
def split_by_product(frame, test_size, seed):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    left, right = next(splitter.split(frame, groups=frame["product_id"]))
    return frame.iloc[left].copy(), frame.iloc[right].copy()


train_val, _heldout = split_by_product(reviews, test_size=0.2, seed=42)
train_full, val_full = split_by_product(train_val, test_size=0.25, seed=43)
train_reviews, _ = train_test_split(
    train_full, train_size=6000, stratify=train_full["label"], random_state=42
)
val_reviews, _ = train_test_split(
    val_full, train_size=1500, stratify=val_full["label"], random_state=43
)

print("Обучение:", len(train_reviews), "Validation:", len(val_reviews))

Обучение: 6000 Validation: 1500


In [6]:
balance = (
    train_reviews["label"].value_counts(normalize=True).sort_index()
    .rename(index=dict(enumerate(LABEL_NAMES)))
    .rename("доля в train")
)
display(balance.to_frame())

,доля в train
label,
отрицательный,0.163833
нейтральный,0.085833
положительный,0.750333


## 1. Первый вариант с TF-IDF

Начнём с простого способа, с которым потом сравним BERT. Представим отзыв числами и передадим их логистической регрессии. Word TF-IDF использует слова и пары соседних слов. Например, сочетание «не подошёл» может быть полезнее каждого слова по отдельности. Слова, которые встречаются почти в каждом отзыве, получают меньший вес.

Словарь строим на train и используем тот же словарь на validation. В таблице выше видно, что положительных отзывов гораздо больше. Поэтому посмотрим на две метрики. Accuracy — доля верных ответов. Macro-F1 усредняет результат по классам и даёт редкому классу такой же вес, как частому.

**Задача.** Соберите первый вариант от признаков до предсказаний на validation. Для `TfidfVectorizer` используйте `ngram_range=(1, 2)`, `min_df=2`, `max_features=50_000` и `sublinear_tf=True`. У логистической регрессии зафиксируйте `C=4.0`, `max_iter=500` и `class_weight="balanced"`. Последний параметр даёт редким классам больший вес при обучении.

[Документация TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) · [Документация LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

<details>
<summary><b>Нужна подсказка?</b></summary>

Посмотрите на методы `fit_transform` и `transform` у `TfidfVectorizer`, а также `fit` и `predict` у `LogisticRegression`. Сначала решите, на какой части данных разрешено обучать словарь и классификатор.

</details>

In [7]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_features=50_000, sublinear_tf=True
)
x_train_word = word_vectorizer.fit_transform(train_reviews["input"])
x_val_word = word_vectorizer.transform(val_reviews["input"])

word_clf = LogisticRegression(max_iter=500, class_weight="balanced", C=4.0)
word_clf.fit(x_train_word, train_reviews["label"])
pred_word = word_clf.predict(x_val_word)

In [8]:
assert x_train_word.shape[0] == len(train_reviews)
assert x_val_word.shape[0] == len(val_reviews)
print("word TF-IDF accuracy:", accuracy_score(val_reviews["label"], pred_word))
print("word TF-IDF macro-F1:", f1_score(val_reviews["label"], pred_word, average="macro"))

word TF-IDF accuracy: 0.8313333333333334
word TF-IDF macro-F1: 0.6315317581428234


### Опечатки и необычное написание

Для словного TF-IDF `очень` и `оооочень` — разные слова. Если брать короткие фрагменты букв, у них найдутся совпадения вроде `чень`. Такие фрагменты называют символьными n-граммами. Проверим, помогут ли они предсказывать рейтинг.

**Задача.** Добавьте к словным признакам символьные и заново обучите классификатор. Для символьных признаков измените `analyzer` на `"char_wb"`, а `ngram_range` на `(3, 5)`. Остальные настройки и выборки оставьте прежними. Сравните обе метрики со словным TF-IDF.

<details>
<summary><b>Нужна подсказка?</b></summary>

У `TfidfVectorizer` есть режим `analyzer="char_wb"`. Разреженные матрицы можно соединять через `hstack`. Проверьте, что в объединяемых матрицах одинаковое число строк.

</details>

In [9]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=2,
    max_features=50_000, sublinear_tf=True,
)
x_train_char = char_vectorizer.fit_transform(train_reviews["input"])
x_val_char = char_vectorizer.transform(val_reviews["input"])

x_train_tfidf = hstack([x_train_word, x_train_char]).tocsr()
x_val_tfidf = hstack([x_val_word, x_val_char]).tocsr()
tfidf_clf = LogisticRegression(max_iter=500, class_weight="balanced", C=4.0)
tfidf_clf.fit(x_train_tfidf, train_reviews["label"])
pred_tfidf = tfidf_clf.predict(x_val_tfidf)

In [10]:
assert x_train_tfidf.shape[1] == x_train_word.shape[1] + x_train_char.shape[1]
print("word + char accuracy:", accuracy_score(val_reviews["label"], pred_tfidf))
print("word + char macro-F1:", f1_score(val_reviews["label"], pred_tfidf, average="macro"))

word + char accuracy: 0.8333333333333334
word + char macro-F1: 0.6296428221795568


## 2. Что готовый BERT знает о тексте

TF-IDF ищет слова и их фрагменты. BERT ещё учитывает, что написано рядом с ними. Прежде чем посмотреть, как он это делает, узнаем, какие части текста получает модель. Эти части называют токенами. Одно слово может состоять из нескольких токенов.

In [11]:
BERT_ID = "cointegrated/rubert-tiny2"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_ID)
bert_mlm = AutoModelForMaskedLM.from_pretrained(BERT_ID).to(DEVICE).eval()
bert_model = bert_mlm.base_model

examples = ["качество", "качетсво", "оооочень", "маломерит", "имбовый"]
for text in examples:
    encoded = bert_tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    pieces = [text[start:end] for start, end in encoded["offset_mapping"]]
    print(f"{text:12} -> {pieces}")

качество     -> ['качество']
качетсво     -> ['каче', 'тс', 'во']
оооочень     -> ['ооо', 'оче', 'нь']
маломерит    -> ['малом', 'ери', 'т']
имбовый      -> ['им', 'бов', 'ый']


При обучении BERT некоторые токены заменяли на `[MASK]` и просили модель их восстановить, глядя на текст слева и справа. Попробуем дать ей два предложения с пропусками. Какие слова подходят в каждом случае?

In [12]:
def predict_mask(text, k=5):
    batch = bert_tokenizer(text, return_tensors="pt").to(DEVICE)
    position = torch.where(
        batch["input_ids"][0] == bert_tokenizer.mask_token_id
    )[0].item()
    with torch.inference_mode():
        logits = bert_mlm(**batch).logits[0, position]
    ids = logits.topk(k).indices
    return [bert_tokenizer.decode([token_id]).strip() for token_id in ids]


print(predict_mask("Курьер привёз [MASK] вечером."))
print(predict_mask("Она подошла к банку, чтобы снять [MASK]."))

['его', 'домой', 'её', 'ее', 'сюда']
['маску', 'деньги', 'квартиру', 'ее', 'его']


### Батч, padding и маска

Подадим модели несколько текстов сразу. Такая группа называется батчем. В батче дополняем короткие тексты токенами `[PAD]` до длины самого длинного. В таблице ниже `attention_mask` равна 0 для добавленных `[PAD]` и 1 для остальных токенов.

In [13]:
batch = bert_tokenizer(
    ["Фильм понравился.", "Фильм оказался намного интереснее, чем я ожидал."],
    padding=True, return_tensors="pt",
)
display(pd.DataFrame({
    "токены": bert_tokenizer.convert_ids_to_tokens(batch["input_ids"][0]),
    "attention_mask": batch["attention_mask"][0].tolist(),
}))

,токены,attention_mask
0,[CLS],1
1,Фильм,1
2,понравился,1
3,.,1
4,[SEP],1
5,[PAD],0
6,[PAD],0
7,[PAD],0
8,[PAD],0
9,[PAD],0


`hidden` хранит по одному контекстному эмбеддингу на токен. В нашем примере два текста, по 11 токенов с учётом `[PAD]` и по 312 чисел в каждом эмбеддинге. Поэтому форма `[B, T, H]` здесь — `[2, 11, 312]`.

Для классификации отзыва нужен один эмбеддинг на весь текст. При mean pooling усредняем эмбеддинги токенов, пропуская `[PAD]`.

Напишите `mean_pool`, которая вернёт тензор формы `[B, H]`. Усредняйте только позиции с маской 1. У `[CLS]` и `[SEP]` маска равна 1, у `[PAD]` — 0.

<details>
<summary><b>Нужна подсказка?</b></summary>

Маска имеет форму `[B, T]`, а hidden states — `[B, T, H]`. Подумайте, как добавить маске одну ось, занулить padding и усреднить только настоящие токены. Пригодятся `unsqueeze`, `sum` и `clamp`.

</details>

In [14]:
def mean_pool(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

In [15]:
batch_on_device = batch.to(DEVICE)
with torch.inference_mode():
    hidden = bert_model(**batch_on_device).last_hidden_state
pooled = mean_pool(hidden, batch_on_device["attention_mask"])

assert pooled.shape == (2, hidden.shape[-1])
print("hidden:", tuple(hidden.shape), "pooled:", tuple(pooled.shape))

# Третья позиция — padding. В среднем должны остаться первые две.
toy_hidden = torch.tensor([[[1.0, 2.0], [3.0, 4.0], [100.0, 100.0]]])
toy_mask = torch.tensor([[1, 1, 0]])
expected = torch.tensor([[2.0, 3.0]])
assert torch.allclose(mean_pool(toy_hidden, toy_mask), expected), "Проверьте, как учтён padding."
print("Проверка пройдена: padding не влияет на среднее.")

hidden: (2, 11, 312) pooled: (2, 312)
Проверка пройдена: padding не влияет на среднее.


In [16]:
def embed_texts(texts, batch_size=None, max_length=128):
    if batch_size is None:
        batch_size = 128 if DEVICE.type == "cuda" else 32
    vectors = []

    for start in tqdm(range(0, len(texts), batch_size), leave=False):
        batch = bert_tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        ).to(DEVICE)
        with torch.inference_mode():
            hidden = bert_model(**batch).last_hidden_state
        pooled = mean_pool(hidden, batch["attention_mask"])
        vectors.append(F.normalize(pooled, dim=1).cpu())

    return torch.cat(vectors)

In [17]:
# Короткий текст получит padding рядом с длинным. Его вектор должен остаться почти тем же.
short = "Фильм оказался неожиданно хорошим."
long = "Сначала фильм казался медленным, но постепенно история раскрылась."
alone = embed_texts([short])[0]
in_batch = embed_texts([short, long])[0]
similarity = F.cosine_similarity(alone, in_batch, dim=0).item()
print(f"Косинусная близость: {similarity:.6f} (ожидаем около 1)")

Косинусная близость: 1.000000 (ожидаем около 1)


Проверим, как BERT справится с той же задачей про рейтинг. Вместо TF-IDF подадим логистической регрессии по одному эмбеддингу на отзыв. Обучим только классификатор. Веса самого BERT остаются прежними, поэтому его называют замороженным.

**Задача.** Обучите логистическую регрессию на `train_bert` с теми же настройками, что и для TF-IDF. Получите предсказания для `val_bert` и сравните метрики.

<details>
<summary><b>Нужна подсказка?</b></summary>

Эмбеддинги — обычные числовые признаки. Scikit-learn принимает NumPy-массивы, поэтому проверьте тип `train_bert` перед передачей в `LogisticRegression`.

</details>

In [18]:
train_bert = embed_texts(train_reviews["input"].tolist())
val_bert = embed_texts(val_reviews["input"].tolist())

frozen_clf = LogisticRegression(max_iter=500, class_weight="balanced", C=4.0)
frozen_clf.fit(train_bert.numpy(), train_reviews["label"])
pred_bert = frozen_clf.predict(val_bert.numpy())

  0%|          | 0/188 [00:00<?, ?it/s]

  2%|▏         | 3/188 [00:00<00:07, 24.25it/s]

  3%|▎         | 6/188 [00:00<00:06, 26.40it/s]

  5%|▍         | 9/188 [00:00<00:06, 27.19it/s]

  7%|▋         | 14/188 [00:00<00:05, 31.31it/s]

 10%|█         | 19/188 [00:00<00:04, 36.99it/s]

... (28 outputs left out)

  9%|▊         | 4/47 [00:00<00:01, 38.67it/s]

 32%|███▏      | 15/47 [00:00<00:00, 57.74it/s]

 55%|█████▌    | 26/47 [00:00<00:00, 75.84it/s]

 81%|████████  | 38/47 [00:00<00:00, 88.35it/s]

In [19]:
print("frozen BERT accuracy:", accuracy_score(val_reviews["label"], pred_bert))
print("frozen BERT macro-F1:", f1_score(val_reviews["label"], pred_bert, average="macro"))

frozen BERT accuracy: 0.73
frozen BERT macro-F1: 0.5918024466326472


Теперь посмотрим на конкретные отзывы. В таблице ниже TF-IDF угадал класс рейтинга, а BERT ошибся. Выберите один отзыв. Что в его тексте могло подсказать правильный класс? Согласны ли вы с классом, который получился из звёзд?

In [20]:
errors = val_reviews[["rating", "input", "label"]].copy()
errors["TF-IDF"] = pred_tfidf
errors["BERT"] = pred_bert
interesting = errors[(errors["TF-IDF"] == errors["label"]) & (errors["BERT"] != errors["label"])].head(6).copy()
for column in ["label", "TF-IDF", "BERT"]:
    interesting[column] = LABEL_NAMES[interesting[column].to_numpy()]
pd.set_option("display.max_colwidth", 220)
display(interesting)

,rating,input,label,TF-IDF,BERT
5545,5,"Футболка оверсайз. Надо иметь это ввиду. Сначала заказала 4xl на свой 56й , сильно утонула. Перезаказала на 2xl, не на много меньше, но оставила. Буду носить с брюками. Просторная. Качество хорошее, плотный трикотаж.",положительный,положительный,нейтральный
14108,5,"Очень понравился и детям и взрослым!) Бабушка даже сказала, что не будет против, если внук оставит воробья ей)) Классный! На одном крылышке рисунок слегка растёрт, но не критично.",положительный,положительный,нейтральный
5955,4,"Сам крем хороший, пользуюсь им давно. Но пришёл почему то в двух пакетах , коробочка до конца не закрыта и слегка мятая, под крышкой почему то нет плёнки, сложилось впечатление, что им пользовались.",положительный,положительный,нейтральный
14574,5,"В коробке 7 блинов, и цена соотвественно… Заглушки не понадобились, залили гипсом, быстро засохло Использовали трубу металопластиковую 2,5 примерно 1м и ещё одну трубу сверху толще и короче чтобы под весом не гнулась...",положительный,положительный,нейтральный
20924,5,"Очень тяжёлые, лакост которые жёлтые, а не прозрачные. Заказывала на пробу думала может повезёт, но нет.От оригинала которые жёлтые ни чем не отличается! Стойкие очень!",положительный,положительный,нейтральный
8164,5,"Тушь бомба, уже третий раз беру. Даже если попадаю под дождь или снег всё ок. Даже зимой после покрытых белых ресниц от холода, снега и дыхания... Всё ок!!))",положительный,положительный,отрицательный


## 3. Как выбрать продолжение истории

В отзывах мы предсказывали класс по одному тексту. Теперь сравним два варианта окончания с одним и тем же началом. В наборе XStoryCloze у каждой истории есть четыре первых предложения и два варианта окончания. Посмотрим, какой способ чаще выбирает окончание из исходной истории.

In [21]:
XSTORY_URL = (
    "https://huggingface.co/datasets/juletxara/xstory_cloze/resolve/main/"
    "spring2016.val.ru.tsv.split_20_80_train.tsv"
)
stories = pd.read_csv(XSTORY_URL, sep="	")
story_columns = [f"input_sentence_{i}" for i in range(1, 5)]
stories["story"] = stories[story_columns].fillna("").agg(" ".join, axis=1)
stories["correct"] = stories["answer_right_ending"] - 1
display(stories[["story", "sentence_quiz1", "sentence_quiz2", "correct"]].head(2))

,story,sentence_quiz1,sentence_quiz2,correct
0,Рик вырос в неблагополучной семье. Он никогда не находил поддержки в семье и подался в банды. Вскоре Рика подстрелили в ходе ограбления. Инцидент заставил его начать с чистого листа.,Теперь он счастлив.,Он вступил в банду.,0
1,"Лаверне нужно что-то приготовить на вечеринку ее подруги. Она решает испечь партию брауни. Она выбирает рецепт и точно его придерживается. Лаверна пробует один из брауни, чтобы убедиться, что он вкусный.","Брауни такие вкусные, что Лаверна съедает два.",Лаверна не идет на вечеринку своей подруги.,0


Сравним историю с каждым окончанием по словам и их фрагментам, как в примере с отзывами. Для этой демонстрации строим словарь по текстам всех историй и обоих вариантов. Правильные ответы понадобятся для подсчёта accuracy. Чтобы оценить работу на новых историях, словарь нужно построить на отдельном наборе.

In [22]:
all_story_texts = pd.concat([
    stories["story"], stories["sentence_quiz1"], stories["sentence_quiz2"]
]).tolist()
story_word = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
story_char = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True
)
all_word = story_word.fit_transform(all_story_texts)
all_char = story_char.fit_transform(all_story_texts)
all_vectors = normalize(hstack([all_word, all_char]).tocsr())
n = len(stories)
story_vectors = all_vectors[:n]
ending1_vectors = all_vectors[n:2 * n]
ending2_vectors = all_vectors[2 * n:]

Косинусная близость показывает, насколько совпадают направления двух векторов. Чем она больше, тем ближе тексты по выбранным признакам.

**Задача.** Для каждой истории посчитайте её близость к первому и ко второму окончанию. Выберите вариант с большей близостью.

<details>
<summary><b>Нужна подсказка?</b></summary>

Векторы уже нормированы: cosine similarity равна их скалярному произведению. Для sparse-матриц пригодятся поэлементный `multiply` и сумма по строкам. Класс `0` означает первое окончание, класс `1` — второе.

</details>

In [23]:
score1 = np.asarray(story_vectors.multiply(ending1_vectors).sum(axis=1)).ravel()
score2 = np.asarray(story_vectors.multiply(ending2_vectors).sum(axis=1)).ravel()
pred_story_tfidf = (score2 > score1).astype(int)

In [24]:
assert set(np.unique(pred_story_tfidf)) <= {0, 1}
print("TF-IDF accuracy:", accuracy_score(stories["correct"], pred_story_tfidf))

TF-IDF accuracy: 0.5388888888888889


Теперь вместо TF-IDF возьмём эмбеддинги BERT и сравним те же пары. Косинусная близость показывает сходство двух текстов. Если поменять их местами, значение будет тем же. Дальше проверим, как GPT учитывает порядок событий.

In [25]:
story_bert = embed_texts(stories["story"].tolist())
ending1_bert = embed_texts(stories["sentence_quiz1"].tolist())
ending2_bert = embed_texts(stories["sentence_quiz2"].tolist())
bert_score1 = (story_bert * ending1_bert).sum(dim=1).numpy()
bert_score2 = (story_bert * ending2_bert).sum(dim=1).numpy()
pred_story_bert = (bert_score2 > bert_score1).astype(int)
print("BERT accuracy:", accuracy_score(stories["correct"], pred_story_bert))

  0%|          | 0/12 [00:00<?, ?it/s]

 17%|█▋        | 2/12 [00:00<00:00, 17.92it/s]

 42%|████▏     | 5/12 [00:00<00:00, 18.92it/s]

100%|██████████| 12/12 [00:00<00:00, 37.22it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

 17%|█▋        | 2/12 [00:00<00:00, 17.47it/s]

... (2 outputs left out)

  0%|          | 0/12 [00:00<?, ?it/s]

 25%|██▌       | 3/12 [00:00<00:00, 25.95it/s]

100%|██████████| 12/12 [00:00<00:00, 55.39it/s]

BERT accuracy: 0.5833333333333334


In [26]:
comparison = stories[["story", "sentence_quiz1", "sentence_quiz2", "correct"]].copy()
comparison["TF-IDF"] = pred_story_tfidf
comparison["BERT"] = pred_story_bert
display(comparison[comparison["TF-IDF"] != comparison["BERT"]].head(5))

,story,sentence_quiz1,sentence_quiz2,correct,TF-IDF,BERT
1,"Лаверне нужно что-то приготовить на вечеринку ее подруги. Она решает испечь партию брауни. Она выбирает рецепт и точно его придерживается. Лаверна пробует один из брауни, чтобы убедиться, что он вкусный.","Брауни такие вкусные, что Лаверна съедает два.",Лаверна не идет на вечеринку своей подруги.,0,1,0
2,"Сара не один год мечтала побывать в Европе. Она наконец-то накопила достаточно для поездки. Она приземлилась в Испании и путешествовала по континенту на восток. Ей не понравилось, насколько все было другим.",После Сара решила перебраться в Европу.,"Сара решила, что ей больше нравится дома, чем в Европе.",1,1,0
3,"Джина боялась, что тесто для печенья из тубуса будет ужасным. Она очень обрадовалась, когда поняла, что ошибалась. Печенье из тубуса было не хуже приготовленного самостоятельно. Джина планировала съесть только 2 пече...","Джине так понравилось печенье, что она съела все за раз.",Джина раздала печенье в своей церкви.,0,1,0
6,Джим нашел старый одноразовый фотоаппарат на дне своего ящика с барахлом. Он начал снимать все вокруг себя. Счетчик прощелкал к последнему фото. Джим начал улавливать серьезность ситуации.,"Джим тщательно обдумал, что хочет снять.",Джим сделал еще 20 фото.,0,1,0
9,"Кэролайн была студенткой медицинского университета. Кэролайн очень тяжело работала, чтобы получать хорошие оценки. Как-то Кэролайн не хватило одного балла, чтобы сдать тест. Кэролайн очень расстроилась, но продолжила...",Но она сдалась.,Позже она сдала тест.,1,1,0


## 4. GPT оценивает продолжение

GPT предсказывает следующий токен по уже прочитанному тексту. Поэтому порядок важен: `начался дождь → открыли зонты` и обратная последовательность получают разные вероятности.

In [27]:
GPT_ID = "ai-forever/rugpt3small_based_on_gpt2"
GPT_DEVICE = torch.device("cpu") if DEVICE.type == "mps" else DEVICE
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_ID)
gpt_model = AutoModelForCausalLM.from_pretrained(GPT_ID).to(GPT_DEVICE).eval()
if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

prompt = "Фильм оказался таким хорошим, что я"
batch = gpt_tokenizer(prompt, return_tensors="pt").to(GPT_DEVICE)
with torch.inference_mode():
    next_logits = gpt_model(**batch).logits[0, -1]
next_ids = next_logits.topk(6).indices
print([gpt_tokenizer.decode([token_id]) for token_id in next_ids])

[' не', ' даже', ' его', ',', ' решил', ' с']


Возьмём историю и одно из готовых окончаний. Для каждого токена окончания посмотрим, какую вероятность GPT дала ему, прочитав всё, что идёт перед ним.

`continuation_scores` берёт средний логарифм вероятности по токенам окончания. Среднее позволяет сравнить варианты разной длины. Выбираем окончание с большей оценкой.

In [28]:
def tokenize_continuations(prompts, endings):
    prompts = [prompt + "\n" for prompt in prompts]
    full_texts = [prompt + ending for prompt, ending in zip(prompts, endings)]
    prompt_lengths = torch.tensor([len(prompt) for prompt in prompts], device=GPT_DEVICE)
    batch = gpt_tokenizer(
        full_texts, padding=True, return_offsets_mapping=True, return_tensors="pt"
    )
    offsets = batch.pop("offset_mapping").to(GPT_DEVICE)
    return batch.to(GPT_DEVICE), offsets, prompt_lengths

In [29]:
def continuation_scores(prompts, endings, batch_size=8):
    scores = []
    for start in tqdm(range(0, len(prompts), batch_size), leave=False):
        batch, offsets, prompt_lengths = tokenize_continuations(
            prompts[start:start + batch_size], endings[start:start + batch_size]
        )
        with torch.inference_mode():
            logits = gpt_model(**batch).logits[:, :-1]
        targets = batch["input_ids"][:, 1:]
        token_log_probs = logits.log_softmax(dim=-1).gather(
            -1, targets.unsqueeze(-1)
        ).squeeze(-1)
        ending_mask = (
            (offsets[:, 1:, 1] > prompt_lengths.unsqueeze(1))
            & batch["attention_mask"][:, 1:].bool()
        )
        score = (token_log_probs * ending_mask).sum(1) / ending_mask.sum(1)
        scores.extend(score.cpu().tolist())
    return np.asarray(scores)

**Задача.** Для каждой истории получите оценки GPT для первого и второго окончания. По этим оценкам выберите один вариант. Посмотрим, как часто GPT совпадает с ответом из набора по сравнению с TF-IDF и BERT.

<details>
<summary><b>Нужна подсказка?</b></summary>

`continuation_scores` принимает два одинаково длинных списка: истории и соответствующие им окончания. Получите два массива оценок и сравните их поэлементно.

</details>

In [30]:
gpt_score1 = continuation_scores(
    stories["story"].tolist(), stories["sentence_quiz1"].tolist()
)
gpt_score2 = continuation_scores(
    stories["story"].tolist(), stories["sentence_quiz2"].tolist()
)
pred_story_gpt = (gpt_score2 > gpt_score1).astype(int)

  0%|          | 0/45 [00:00<?, ?it/s]

  2%|▏         | 1/45 [00:00<00:08,  5.32it/s]

  4%|▍         | 2/45 [00:00<00:07,  5.87it/s]

  7%|▋         | 3/45 [00:00<00:07,  5.95it/s]

  9%|▉         | 4/45 [00:00<00:06,  6.16it/s]

 11%|█         | 5/45 [00:00<00:06,  5.83it/s]

... (82 outputs left out)

 93%|█████████▎| 42/45 [00:07<00:00,  5.71it/s]

 96%|█████████▌| 43/45 [00:07<00:00,  5.62it/s]

 98%|█████████▊| 44/45 [00:07<00:00,  5.83it/s]

100%|██████████| 45/45 [00:08<00:00,  5.79it/s]

In [31]:
story_results = pd.DataFrame({
    "способ": ["TF-IDF", "BERT", "GPT"],
    "accuracy": [
        accuracy_score(stories["correct"], pred_story_tfidf),
        accuracy_score(stories["correct"], pred_story_bert),
        accuracy_score(stories["correct"], pred_story_gpt),
    ],
})
display(story_results.round(3))

,способ,accuracy
0,TF-IDF,0.539
1,BERT,0.583
2,GPT,0.583


## 5. GPT пишет продолжение

До этого GPT выбирала из готовых вариантов. Теперь дадим ей начало истории и попросим написать продолжение.

При низкой `temperature` модель чаще выбирает самые вероятные токены. При высокой температуре менее вероятные варианты получают больше шансов. `top_p=0.9` оставляет столько самых вероятных токенов, сколько нужно, чтобы их суммарная вероятность была хотя бы 0.9.

Проверим, как температура влияет на продолжение одной и той же истории.

GPT выбирает токены случайно, а seed задаёт начальное состояние этого выбора. Внутри пары используем один и тот же seed, чтобы менять только температуру. С тремя разными seed проверяем, повторяется ли замеченная разница при других случайных продолжениях.

**Что написать.** Для каждого seed из `generation_seeds` получите два продолжения для `example_story` через `generate_ending`. Сделайте один вызов с `temperature=0.7` и второй с `temperature=1.2`. В обоих оставьте `top_p=0.9` и используйте один и тот же seed. Выведите продолжения парами и подпишите seed и температуру.

**Что сравнить.** Прочитайте начало истории и оба продолжения в каждой паре. Что изменилось при повышении температуры? Сравните, насколько тексты связаны с началом и есть ли в них повторы. Приведите один пример и проверьте, видно ли то же в других парах.

In [32]:
def generate_ending(prompt, temperature, top_p=0.9, max_new_tokens=45):
    batch = gpt_tokenizer(prompt, return_tensors="pt").to(GPT_DEVICE)
    with torch.inference_mode():
        output = gpt_model.generate(
            **batch, do_sample=True, temperature=temperature, top_p=top_p,
            max_new_tokens=max_new_tokens, pad_token_id=gpt_tokenizer.eos_token_id,
        )
    generated = output[0, batch["input_ids"].shape[1]:]
    return gpt_tokenizer.decode(generated, skip_special_tokens=True).strip()

<details>
<summary><b>Нужна подсказка?</b></summary>

Перед каждым вызовом `generate_ending` задайте seed через `set_seed`. Для первой пары задайте 42 перед генерацией при 0.7 и снова 42 перед генерацией при 1.2.

</details>

In [33]:
example_story = stories.iloc[1]["story"]
print(f"Начало истории\n{example_story}\n")
generation_seeds = [42, 43, 44]

for seed in generation_seeds:
    print(f"seed={seed}")
    for temperature in [0.7, 1.2]:
        set_seed(seed)
        ending = generate_ending(example_story, temperature=temperature, top_p=0.9)
        print(f"temperature={temperature}: {ending}\n")

Начало истории
Лаверне нужно что-то приготовить на вечеринку ее подруги. Она решает испечь партию брауни. Она выбирает рецепт и точно его придерживается. Лаверна пробует один из брауни, чтобы убедиться, что он вкусный.

seed=42
temperature=0.7: Она берет его и кладет на тарелку.

— Вкусно, — говорит она.

Она ставит брауни на стол.

— Я не могу его съесть, — говорит

temperature=1.2: Затем берет ложку и кладет на край формы с глазурью. Подходит ко всем остальным. Заправка не самая плохая. Лаверна ставит брауни на огонь. Он готов. Все кладет в тарелку

seed=43
temperature=0.7: Она вынимает его из духовки и начинает резать на кусочки.

– Что это? – спрашивает она у сестры.

– Кексы, – отвечает Лаверна.

–

temperature=1.2: Лаверна готовится медленно и внимательно. В середине процесса она делает глубокую глубокую тарелку. Перед тем как приступить к пиршеству, ей хочется сделать два небольших бутерброда на две порционные тарелки — одну размером в ладонь

seed=44
temperature=0.7: Лаверна ст

## Итог

На отзывах сравнили TF-IDF и готовые эмбеддинги BERT. На историях сравнили сходство двух текстов и вероятность продолжения у GPT. На практике вернёмся к отзывам и проверим, что изменится, если обучать BERT на наших примерах.

## Бонус для дома

1. Придумайте дополнительные признаки по тексту. Это могут быть особенности оформления, длина, частоты выбранных символов или любые другие проверяемые идеи. Объясните гипотезу, добавьте признаки к TF-IDF и сравните validation macro-F1.
2. Найдите пять историй, где GPT наиболее уверенно выбрала неправильное окончание.
3. Для 40 историй найдите окончание среди всех 40 кандидатов и сохраните ответы в CSV. Посчитайте, сколько окончаний выбрали две или больше историй.

<details>
<summary><b>Нужна подсказка?</b></summary>

Для работы со строками могут пригодиться `count`, `len`, `isupper`, `isalpha` и регулярные выражения. Это не список обязательных признаков: сначала сформулируйте собственную гипотезу.

</details>

In [34]:
# Один из возможных вариантов, а не обязательный набор признаков.
ELONGATED = re.compile(r"([а-яёa-z])\1{2,}", flags=re.IGNORECASE)

def style_features(texts):
    rows = []
    for text in texts:
        letters = [char for char in text if char.isalpha()]
        rows.append([
            text.count("!"),
            sum(char.isupper() for char in letters) / max(len(letters), 1),
            int(ELONGATED.search(text) is not None),
        ])
    return np.asarray(rows, dtype="float32")

<details>
<summary><b>Нужна подсказка?</b></summary>

`style_features` должна вернуть двумерную матрицу: одна строка на текст. Её можно присоединить к TF-IDF через `hstack`. Новый классификатор сравнивайте с прежним на той же validation-выборке.

</details>

In [35]:
train_extra = style_features(train_reviews["input"])
val_extra = style_features(val_reviews["input"])
x_train_extra = hstack([x_train_tfidf, train_extra]).tocsr()
x_val_extra = hstack([x_val_tfidf, val_extra]).tocsr()

extra_clf = LogisticRegression(max_iter=500, class_weight="balanced", C=4.0)
extra_clf.fit(x_train_extra, train_reviews["label"])
pred_extra = extra_clf.predict(x_val_extra)

base_f1 = f1_score(val_reviews["label"], pred_tfidf, average="macro")
extra_f1 = f1_score(val_reviews["label"], pred_extra, average="macro")
print("word + char:", base_f1)
print("word + char + дополнительные признаки:", extra_f1)
print("изменение:", extra_f1 - base_f1)

word + char: 0.6296428221795568
word + char + дополнительные признаки: 0.6247569472907615
изменение: -0.004885874888795327


<details>
<summary><b>Нужна подсказка?</b></summary>

Сначала оставьте строки, где prediction не равен правильному ответу. Затем для каждой такой строки найдите разность между score выбранного неправильного и правильного окончания и отсортируйте ошибки по этой разности.

</details>

In [36]:
correct_score = np.where(stories["correct"] == 0, gpt_score1, gpt_score2)
wrong_score = np.where(stories["correct"] == 0, gpt_score2, gpt_score1)
gpt_errors = stories[pred_story_gpt != stories["correct"]].copy()
is_error = pred_story_gpt != stories["correct"]
gpt_errors["преимущество ошибки"] = (wrong_score - correct_score)[is_error]
display(gpt_errors.sort_values("преимущество ошибки", ascending=False).head(5))

,story_id,input_sentence_1,input_sentence_2,input_sentence_3,input_sentence_4,sentence_quiz1,sentence_quiz2,answer_right_ending,story,correct,преимущество ошибки
265,ff6a9be8-35fd-4371-9337-0a487aa6bcbd,"Рекс скучал по своим школьным денькам, когда он был гимнастом.","Теперь он боялся, что навсегда потерял свои спортивные навыки.",Однако друг подталкивал его попробовать сделать кувырок колесом на пляже.,Рекс сделал потрясающий кувырок колесом с первой же попытки!,Он успокоился.,В тот день Рекс впервые сделал кувырок колесом.,1,"Рекс скучал по своим школьным денькам, когда он был гимнастом. Теперь он боялся, что навсегда потерял свои спортивные навыки. Однако друг подталкивал его попробовать сделать кувырок колесом на пляже. Рекс сделал потр...",0,4.856402
263,0f1277f9-f7eb-4133-a067-8012ec4ce4c4,Джон сделал большой глоток и проглотил кубик льда.,"Когда кубик льда застрял у него в горле, Джон запаниковал.","Он подогрел воду, чтобы растопить этот кубик льда.","Он быстро хлебнул, и лед растаял.",Джон успокоился.,"Джон очень расстроился, что лед растаял.",1,"Джон сделал большой глоток и проглотил кубик льда. Когда кубик льда застрял у него в горле, Джон запаниковал. Он подогрел воду, чтобы растопить этот кубик льда. Он быстро хлебнул, и лед растаял.",0,3.559707
27,d4aff04b-ba93-4cff-8b35-7eb10dd9e23f,Рон смотрел погоню на высоких скоростях по ТВ.,Коп гнался за плохим парнем по его району.,Это было жутко!,Но коп поймал плохого парня.,Рон начал собирать бутылочные пробки.,Рон похлопал работе офицера.,2,Рон смотрел погоню на высоких скоростях по ТВ. Коп гнался за плохим парнем по его району. Это было жутко! Но коп поймал плохого парня.,1,2.784692
211,8f38419d-c80d-40b0-8770-3a8a86e4a5ad,Джейн нужно было подзаработать.,Она вспомнила об оплачиваемых психологических экспериментах в колледже.,Джейн поискала оплачиваемые эксперименты и нашла сервис Mechanical Turk.,"Теперь она могла участвовать в психологических экспериментах, находясь дома.",Джейн решила больше не участвовать в психологических экспериментах.,Джейн наконец удалось подзаработать.,2,Джейн нужно было подзаработать. Она вспомнила об оплачиваемых психологических экспериментах в колледже. Джейн поискала оплачиваемые эксперименты и нашла сервис Mechanical Turk. Теперь она могла участвовать в психолог...,1,2.577757
52,28f02d77-17f2-4ae0-bac9-3d6b48603662,Часы Джона были сломаны некоторое время.,"Он не собрал денег, чтобы отремонтировать их.",Его жена решила отдать их в ремонт и заплатить на них.,"Она отдала их ему, когда он пришел домой с работы.",Жена очень расстроила Джона.,Джон очень обрадовался.,2,"Часы Джона были сломаны некоторое время. Он не собрал денег, чтобы отремонтировать их. Его жена решила отдать их в ремонт и заплатить на них. Она отдала их ему, когда он пришел домой с работы.",1,2.480007


<details>
<summary><b>Нужна подсказка?</b></summary>

Матричное произведение эмбеддингов историй на транспонированную матрицу окончаний даст все попарные близости. Дальше пригодятся `argmax`, `DataFrame` и `to_csv`. Чтобы найти повторы, посчитайте, сколько раз встречается каждый `ending_id`.

</details>

In [37]:
n = 40
correct_endings = np.where(
    stories["correct"].to_numpy() == 0,
    stories["sentence_quiz1"].to_numpy(),
    stories["sentence_quiz2"].to_numpy(),
)
left = embed_texts(stories["story"].iloc[:n].tolist())
right = embed_texts(correct_endings[:n].tolist())

score_matrix = left @ right.T
predicted_ending = score_matrix.argmax(dim=1).numpy()
submission = pd.DataFrame({"story_id": np.arange(n), "ending_id": predicted_ending})
submission.to_csv("bonus_submission.csv", index=False)
print("many-candidate accuracy:", np.mean(predicted_ending == np.arange(n)))
ending_counts = submission["ending_id"].value_counts()
reused_endings = (ending_counts > 1).sum()
print("Окончаний, выбранных несколькими историями:", reused_endings)
display(submission.head())

many-candidate accuracy: 0.325
Окончаний, выбранных несколькими историями: 9


,story_id,ending_id
0,0,32
1,1,1
2,2,24
3,3,20
4,4,31


### Если каждое окончание можно использовать только один раз

Если выбирать лучшее окончание для каждой истории отдельно, несколько историй могут выбрать один вариант. Чтобы избежать повторов, можно упорядочить все пары по близости и брать лучшие пары, у которых история и окончание пока свободны. А если нужно найти лучший набор пар по суммарной близости, это задача о назначениях.

Полезные ссылки:

- [Padding и truncation](https://huggingface.co/docs/transformers/main/pad_truncation)
- [Masked language modeling](https://huggingface.co/docs/transformers/tasks/masked_language_modeling)
- [Стратегии генерации](https://huggingface.co/docs/transformers/generation_strategies)
- [Model card ruBERT-tiny2](https://huggingface.co/cointegrated/rubert-tiny2)
- [Model card ruGPT-3 small](https://huggingface.co/ai-forever/rugpt3small_based_on_gpt2)